In [1]:
import jax
import jax.numpy as jnp
from optax.losses import sigmoid_binary_cross_entropy

In [2]:
SEED = 42
key = jax.random.PRNGKey(SEED)

k1, k2, k3 = jax.random.split(key, 3)

In [3]:
params = {
    "W1": jax.random.normal(k1, (2, 2)),
    "b1": jnp.zeros((2,)),

    "W2": jax.random.normal(k2, (2, 2)),
    "b2": jnp.zeros((2,)),

    "W3": jax.random.normal(k3, (2, 1)),
    "b3": jnp.zeros((1,))
}

In [4]:
params

{'W1': Array([[ 0.07592554, -0.48634264],
        [ 1.2903206 ,  0.5196119 ]], dtype=float32),
 'b1': Array([0., 0.], dtype=float32),
 'W2': Array([[ 0.60576403,  0.7990441 ],
        [-0.908927  , -0.63525754]], dtype=float32),
 'b2': Array([0., 0.], dtype=float32),
 'W3': Array([[0.4323065],
        [0.5872638]], dtype=float32),
 'b3': Array([0.], dtype=float32)}

In [5]:
def forward(params, X):
    z1 = X @ params["W1"] + params["b1"]
    a1 = jax.nn.relu(z1)

    z2 = a1 @ params["W2"] + params["b2"]
    a2 = jax.nn.relu(z2)

    z3 = a2 @ params["W3"] + params["b3"]

    return z3

In [10]:
# grads
def loss_fn(params, X, y):
    logits = forward(params, X)

    loss = sigmoid_binary_cross_entropy(
        logits=logits,
        labels=y
    )

    return loss.mean()

grad_fn = jax.grad(sigmoid_binary_cross_entropy)

In [11]:
grad_fn = jax.grad(loss_fn)

In [12]:
X = jnp.array([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
])

y = jnp.array([
    [0.],
    [1.],
    [1.],
    [0.]
])

In [19]:
for epoch in range(10000):

    loss = loss_fn(params, X, y)

    grads = jax.grad(loss_fn)(params, X, y)

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:<12} | Loss: {loss:>12.3f}")

    params = jax.tree.map(
        lambda p, g: p - 1 * g,
        params,
        grads
    )

    # print("W3 depois:")
    # print(params["W3"])

Epoch 0            | Loss:        0.693
Epoch 1000         | Loss:        0.693


Exception ignored in: <function _xla_gc_callback at 0x7f22042844c0>
Traceback (most recent call last):
  File "/home/agent_smith/Documents/.env/lib/python3.10/site-packages/jax/_src/lib/__init__.py", line 118, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 

KeyboardInterrupt



In [15]:
jax.nn.sigmoid(forward(params, X))

Array([[0.49999005],
       [0.49999005],
       [0.49999005],
       [0.49999005]], dtype=float32)

In [16]:
grads = jax.grad(loss_fn)(params, X, y)

In [17]:
grads

{'W1': Array([[0., 0.],
        [0., 0.]], dtype=float32),
 'W2': Array([[0., 0.],
        [0., 0.]], dtype=float32),
 'W3': Array([[0.],
        [0.]], dtype=float32),
 'b1': Array([0., 0.], dtype=float32),
 'b2': Array([0., 0.], dtype=float32),
 'b3': Array([-9.924173e-06], dtype=float32)}

In [37]:
import jax
import jax.numpy as jnp
from optax.losses import sigmoid_binary_cross_entropy

SEED = 42
key = jax.random.PRNGKey(SEED)
k1, k2, k3 = jax.random.split(key, 3)

params = {
    "W1": jax.random.normal(k1, (2, 4)),
    "b1": jnp.zeros((4,)),

    "W2": jax.random.normal(k2, (4, 4)),
    "b2": jnp.zeros((4,)),

    "W3": jax.random.normal(k3, (4, 1)),
    "b3": jnp.zeros((1,))
}

def forward(params, X):
    z1 = X @ params["W1"] + params["b1"]
    a1 = jax.nn.leaky_relu(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    a2 = jax.nn.leaky_relu(z2)
    z3 = a2 @ params["W3"] + params["b3"]
    return z3

def loss_fn(params, X, y):
    logits = forward(params, X)
    return sigmoid_binary_cross_entropy(logits=logits, labels=y).mean()

@jax.jit
def train_step(params, X, y, lr):
    loss, grads = grad_fn(params, X, y)

    params = jax.tree.map(
        lambda p, g: p - lr * g,
        params,
        grads
    )

    return params, loss

grad_fn = jax.value_and_grad(loss_fn)

X = jnp.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = jnp.array([[0.], [1.], [1.], [0.]])

LR = 0.01

for epoch in range(5000):
    params, loss = train_step(params, X, y, LR)

    if epoch % 500 == 0:
        preds = jax.nn.sigmoid(forward(params, X))
        print(f"Epoch {epoch:4d} | loss: {float(loss):.4f}")

Epoch    0 | loss: 0.6762
Epoch  500 | loss: 0.5034
Epoch 1000 | loss: 0.4303
Epoch 1500 | loss: 0.3974
Epoch 2000 | loss: 0.3824
Epoch 2500 | loss: 0.3736
Epoch 3000 | loss: 0.3686
Epoch 3500 | loss: 0.3655
Epoch 4000 | loss: 0.3632
Epoch 4500 | loss: 0.3617


In [38]:
jax.nn.sigmoid(forward(params, X))

Array([[0.02272617],
       [0.48223338],
       [0.9939456 ],
       [0.4954962 ]], dtype=float32)

In [21]:
print(jax.devices())

[CudaDevice(id=0)]


In [22]:
print(jax.default_backend())

gpu


In [25]:
print(y.devices())

{CudaDevice(id=0)}


In [26]:
print(X.devices())

{CudaDevice(id=0)}


In [39]:
grad_fn(params, X, y)

(Array(0.36064202, dtype=float32),
 {'W1': Array([[-4.6845106e-04, -1.0480892e-04, -1.9028154e-04, -6.2950235e-04],
         [-2.4084514e-04,  5.4397387e-06, -1.2122304e-04,  3.9025166e-03]],      dtype=float32),
  'W2': Array([[-0.00022874, -0.00013107,  0.00031688, -0.00048522],
         [-0.00092208, -0.00052835,  0.00127737,  0.00046917],
         [-0.00065561, -0.00037567,  0.00090822, -0.00126719],
         [-0.00189405, -0.0010853 ,  0.00262386, -0.00065488]],      dtype=float32),
  'W3': Array([[-0.00300936],
         [-0.00089503],
         [ 0.00654017],
         [ 0.00361549]], dtype=float32),
  'b1': Array([ 0.0185096 , -0.00053343,  0.00936178,  0.01741739], dtype=float32),
  'b2': Array([ 0.00633866,  0.00363207, -0.00878103, -0.00434976], dtype=float32),
  'b3': Array([-0.00139964], dtype=float32)})

In [40]:
def f(x):
    return x**2

df = jax.grad(f)

In [45]:
df(4.)

Array(8., dtype=float32, weak_type=True)

In [46]:
f(4)

16

In [5]:
import jax
import jax.numpy as jnp

import flax.linen as nn
import optax

In [6]:
class MLP(nn.Module):

    @nn.compact
    def __call__(self, x):

        x = nn.Dense(4)(x)
        x = nn.leaky_relu(x)

        x = nn.Dense(4)(x)
        x = nn.leaky_relu(x)

        x = nn.Dense(1)(x)

        return x

In [7]:
X = jnp.array([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
])

y = jnp.array([
    [0.],
    [1.],
    [1.],
    [0.]
])

In [8]:
model = MLP()

key = jax.random.PRNGKey(42)

params = model.init(
    key,
    X
)["params"]

In [9]:
model

MLP()

In [11]:
print(params.keys())

dict_keys(['Dense_0', 'Dense_1', 'Dense_2'])


In [15]:
params['Dense_0']

{'kernel': Array([[-0.3691293 , -0.898655  ,  0.90054697,  1.4181342 ],
        [-1.1427469 , -0.5788182 , -0.70497435, -0.7854578 ]],      dtype=float32),
 'bias': Array([0., 0., 0., 0.], dtype=float32)}

In [32]:
optimizer = optax.sgd(
    learning_rate=0.01,
    momentum=0.9
)

opt_state = optimizer.init(params)

In [33]:
optimizer

GradientTransformationExtraArgs(init=<function chain.<locals>.init_fn at 0x7f4afc380670>, update=<function chain.<locals>.update_fn at 0x7f4afc380310>)

In [34]:
opt_state

(TraceState(trace={'Dense_0': {'bias': Array([0., 0., 0., 0.], dtype=float32), 'kernel': Array([[0., 0., 0., 0.],
        [0., 0., 0., 0.]], dtype=float32)}, 'Dense_1': {'bias': Array([0., 0., 0., 0.], dtype=float32), 'kernel': Array([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]], dtype=float32)}, 'Dense_2': {'bias': Array([0.], dtype=float32), 'kernel': Array([[0.],
        [0.],
        [0.],
        [0.]], dtype=float32)}}),
 EmptyState())

In [35]:
def loss_fn(params, X, y):

    logits = model.apply(
        {"params": params},
        X
    )

    loss = optax.losses.sigmoid_binary_cross_entropy(
        logits=logits,
        labels=y
    )

    return loss.mean()

In [36]:
@jax.jit
def train_step(params, opt_state, X, y):

    loss, grads = jax.value_and_grad(
        loss_fn
    )(params, X, y)

    updates, opt_state = optimizer.update(
        grads,
        opt_state,
        params
    )

    params = optax.apply_updates(
        params,
        updates
    )

    return params, opt_state, loss

In [37]:
for epoch in range(5000):

    params, opt_state, loss = train_step(
        params,
        opt_state,
        X,
        y
    )

    if epoch % 500 == 0:

        logits = model.apply(
            {"params": params},
            X
        )

        preds = jax.nn.sigmoid(logits)

        print(
            f"Epoch {epoch:4d}"
            f" | Loss: {float(loss):.4f}"
        )

        print(preds.ravel())

Epoch    0 | Loss: 0.0174
[0.01882526 0.97081953 0.991848   0.01264498]
Epoch  500 | Loss: 0.0038
[0.0036583  0.9931766  0.99817824 0.00290907]
Epoch 1000 | Loss: 0.0019
[0.00175544 0.9964712  0.9990891  0.00149001]
Epoch 1500 | Loss: 0.0012
[1.0994615e-03 9.9768806e-01 9.9941498e-01 9.6956163e-04]
Epoch 2000 | Loss: 0.0009
[7.8084023e-04 9.9830663e-01 9.9957758e-01 7.0766360e-04]
Epoch 2500 | Loss: 0.0007
[5.9621513e-04 9.9867505e-01 9.9967283e-01 5.5141310e-04]
Epoch 3000 | Loss: 0.0006
[4.7786458e-04 9.9891859e-01 9.9973553e-01 4.4924393e-04]
Epoch 3500 | Loss: 0.0005
[3.9578081e-04 9.9909008e-01 9.9977899e-01 3.7707636e-04]
Epoch 4000 | Loss: 0.0004
[3.3602523e-04 9.9921703e-01 9.9981105e-01 3.2386847e-04]
Epoch 4500 | Loss: 0.0004
[2.9081112e-04 9.9931443e-01 9.9983537e-01 2.8306121e-04]


In [38]:
from pprint import pprint

pprint(
    jax.tree.map(
        lambda x: x.shape,
        params
    )
)

{'Dense_0': {'bias': (4,), 'kernel': (2, 4)},
 'Dense_1': {'bias': (4,), 'kernel': (4, 4)},
 'Dense_2': {'bias': (1,), 'kernel': (4, 1)}}


In [39]:
loss, grads = jax.value_and_grad(
    loss_fn
)(
    params,
    X,
    y
)

In [40]:
opt_state

(TraceState(trace={'Dense_0': {'bias': Array([-0.00058391, -0.00071849, -0.00241548, -0.00086031], dtype=float32), 'kernel': Array([[-5.7202637e-06, -7.4425202e-07, -1.1927193e-03, -2.1827563e-03],
        [-3.0989671e-05,  2.9703586e-05,  7.7410717e-04,  1.5441352e-03]],      dtype=float32)}, 'Dense_1': {'bias': Array([-1.3533197e-07, -3.5588522e-03, -4.1424180e-04,  5.5386772e-04],      dtype=float32), 'kernel': Array([[ 5.1814311e-08,  4.1657011e-04, -5.3421099e-04,  6.3983141e-05],
        [ 5.6798307e-08,  4.3871341e-04, -5.9744203e-04,  7.9011712e-05],
        [ 1.8715645e-07,  1.6217924e-03, -1.9799545e-03, -5.6023762e-04],
        [-1.6592485e-07,  1.6007548e-04,  1.7966881e-03, -2.2567462e-03]],      dtype=float32)}, 'Dense_2': {'bias': Array([-0.00062026], dtype=float32), 'kernel': Array([[-2.5920940e-06],
        [-3.9434307e-03],
        [ 2.9764143e-03],
        [-2.4072800e-03]], dtype=float32)}}),
 EmptyState())

In [41]:
updates

{'Dense_0': {'bias': Array([-1.70537023e-05,  5.49971264e-05, -1.21955665e-04,  1.67944163e-04],      dtype=float32),
  'kernel': Array([[ 1.8404451e-07,  2.0358513e-08,  2.5878855e-05,  1.2611647e-04],
         [ 7.9370955e-07, -8.0894131e-07, -3.4226521e-04, -2.6379962e-06]],      dtype=float32)},
 'Dense_1': {'bias': Array([ 3.1236094e-09,  5.7839196e-05, -2.4658852e-05, -2.3624174e-05],      dtype=float32),
  'kernel': Array([[-1.7174785e-09, -1.9080817e-05,  1.3558373e-05,  6.3415831e-07],
         [-3.3713698e-09, -3.6366007e-05,  2.6614771e-05,  1.8699083e-07],
         [-1.0182505e-08, -1.2997846e-04,  8.0384249e-05,  2.0127845e-05],
         [ 7.3405957e-09, -9.1523580e-06, -5.7949219e-05,  8.5384323e-05]],      dtype=float32)},
 'Dense_2': {'bias': Array([1.4851091e-05], dtype=float32),
  'kernel': Array([[ 2.0081970e-07],
         [ 1.3261128e-04],
         [-1.1586632e-04],
         [ 9.1169066e-05]], dtype=float32)}}

In [42]:
grads

{'Dense_0': {'bias': Array([-1.1837828e-04,  5.7571738e-06, -4.0620012e-04,  4.6126061e-04],      dtype=float32),
  'kernel': Array([[-6.5363128e-07, -1.4528410e-07, -1.0899243e-04,  2.6781001e-04],
         [-1.6960051e-06,  2.8373313e-06, -2.4447928e-04,  6.3726061e-04]],      dtype=float32)},
 'Dense_1': {'bias': Array([-1.3502974e-08, -4.5842270e-04,  1.4286667e-04,  6.8701222e-05],      dtype=float32),
  'kernel': Array([[ 5.1695053e-09,  4.1653807e-06, -5.4695374e-05, -1.2347095e-06],
         [ 5.6662004e-09,  1.6364497e-06, -5.9950606e-05, -6.4231040e-07],
         [ 1.8677081e-08,  2.5939357e-06, -1.9761077e-04, -3.2058015e-05],
         [-1.6552205e-08,  7.9275096e-06,  1.7512876e-04, -2.2670567e-04]],      dtype=float32)},
 'Dense_2': {'bias': Array([-6.188665e-05], dtype=float32),
  'kernel': Array([[-2.5876014e-07],
         [-3.9350492e-04],
         [ 2.9701792e-04],
         [-2.4016398e-04]], dtype=float32)}}

In [43]:
loss, grads = jax.value_and_grad(
    loss_fn
)(params, X, y)

updates, opt_state = optimizer.update(
    grads,
    opt_state,
    params
)

params = optax.apply_updates(
    params,
    updates
)

In [44]:
updates

{'Dense_0': {'bias': Array([6.4390115e-06, 6.4088536e-06, 2.5801313e-05, 3.1301483e-06],      dtype=float32),
  'kernel': Array([[ 5.8018681e-08,  8.1511091e-09,  1.1824398e-05,  1.6966706e-05],
         [ 2.9586707e-07, -2.9570558e-07, -4.5221718e-06, -2.0269821e-05]],      dtype=float32)},
 'Dense_1': {'bias': Array([ 1.3530175e-09,  3.6613892e-05,  2.2995096e-06, -5.6718213e-06],      dtype=float32),
  'kernel': Array([[-5.1802379e-10, -3.7907846e-06,  5.3548524e-06, -5.6350115e-07],
         [-5.6784671e-10, -3.9647853e-06,  5.9764843e-06, -7.0468229e-07],
         [-1.8711788e-09, -1.4622072e-05,  1.9795698e-05,  5.3627182e-06],
         [ 1.6588455e-09, -1.5199543e-06, -1.7921480e-05,  2.2577771e-05]],      dtype=float32)},
 'Dense_2': {'bias': Array([6.2011995e-06], dtype=float32),
  'kernel': Array([[ 2.5916446e-08],
         [ 3.9425926e-05],
         [-2.9757904e-05],
         [ 2.4067158e-05]], dtype=float32)}}

In [45]:
opt_state

(TraceState(trace={'Dense_0': {'bias': Array([-0.0006439 , -0.00064089, -0.00258013, -0.00031301], dtype=float32), 'kernel': Array([[-5.8018682e-06, -8.1511092e-07, -1.1824399e-03, -1.6966707e-03],
        [-2.9586708e-05,  2.9570558e-05,  4.5221718e-04,  2.0269821e-03]],      dtype=float32)}, 'Dense_1': {'bias': Array([-1.3530175e-07, -3.6613895e-03, -2.2995095e-04,  5.6718214e-04],      dtype=float32), 'kernel': Array([[ 5.1802381e-08,  3.7907847e-04, -5.3548528e-04,  5.6350116e-05],
        [ 5.6784675e-08,  3.9647851e-04, -5.9764844e-04,  7.0468232e-05],
        [ 1.8711788e-07,  1.4622072e-03, -1.9795699e-03, -5.3627184e-04],
        [-1.6588456e-07,  1.5199542e-04,  1.7921480e-03, -2.2577771e-03]],      dtype=float32)}, 'Dense_2': {'bias': Array([-0.00062012], dtype=float32), 'kernel': Array([[-2.5916447e-06],
        [-3.9425925e-03],
        [ 2.9757905e-03],
        [-2.4067159e-03]], dtype=float32)}}),
 EmptyState())

In [47]:
params

{'Dense_0': {'bias': Array([0.363759  , 0.40981442, 1.5411912 , 0.07825339], dtype=float32),
  'kernel': Array([[-0.36527112, -0.8998347 ,  1.6342893 ,  2.2399342 ],
         [-1.118864  , -0.59588563, -1.5411863 , -2.3181846 ]],      dtype=float32)},
 'Dense_1': {'bias': Array([-4.4791464e-05,  2.5758169e+00, -2.2904389e-02, -4.6497315e-01],      dtype=float32),
  'kernel': Array([[ 1.1179107 ,  0.6444629 ,  0.8027978 , -0.2387224 ],
         [ 0.28050026, -0.7113161 , -0.03033422, -0.22242852],
         [-0.69403577, -1.5764734 ,  2.0962355 ,  0.28141844],
         [ 0.00316051, -1.1391668 , -1.290925  ,  2.6739876 ]],      dtype=float32)},
 'Dense_2': {'bias': Array([-0.454116], dtype=float32),
  'kernel': Array([[ 0.02181891],
         [ 3.030232  ],
         [-2.3085518 ],
         [ 2.692896  ]], dtype=float32)}}